# Smart Hospitality Management System
## Module 3: Room Demand Forecasting

End-to-end pipeline: build time series -> EDA -> feature engineering -> preprocessing -> baseline models -> LSTM deep learning model.

## 1. Build Daily Demand Time Series

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 1: Build the daily demand time series from booking-level data
"""
import pandas as pd
import numpy as np

df = pd.read_csv('data/hotel_booking_cleaned.csv')
print(f"Booking-level rows: {df.shape}")

# ------------------------------------------------------------------
# 1. Reconstruct the actual arrival date
# ------------------------------------------------------------------
month_map = {m: i+1 for i, m in enumerate(
    ['January','February','March','April','May','June','July',
     'August','September','October','November','December'])}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)

df['arrival_date'] = pd.to_datetime(dict(
    year=df['arrival_date_year'],
    month=df['arrival_month_num'],
    day=df['arrival_date_day_of_month']
), errors='coerce')

before = len(df)
df = df.dropna(subset=['arrival_date'])
print(f"Dropped {before - len(df)} rows with invalid dates")

# ------------------------------------------------------------------
# 2. Demand = actual arrivals only (exclude cancellations - a cancelled
#    booking never consumed a room on the arrival date)
# ------------------------------------------------------------------
actual_stays = df[df['is_canceled'] == 0].copy()
print(f"Actual (non-cancelled) stays: {len(actual_stays)}")

# ------------------------------------------------------------------
# 3. Aggregate to daily demand, per hotel type
# ------------------------------------------------------------------
daily_demand = (actual_stays.groupby(['arrival_date', 'hotel'])
                 .size()
                 .reset_index(name='room_demand'))

# pivot so each hotel type is its own series, then also build a combined total
pivot = daily_demand.pivot(index='arrival_date', columns='hotel', values='room_demand').fillna(0)
pivot['Total'] = pivot.sum(axis=1)
pivot = pivot.reset_index().sort_values('arrival_date')

# fill any missing calendar dates (days with zero arrivals) with 0
full_range = pd.date_range(pivot['arrival_date'].min(), pivot['arrival_date'].max(), freq='D')
pivot = pivot.set_index('arrival_date').reindex(full_range).fillna(0).rename_axis('date').reset_index()

print(f"\nDaily demand series shape: {pivot.shape}")
print(f"Date range: {pivot['date'].min()} to {pivot['date'].max()}")
print(pivot.head())

pivot.to_csv('data/daily_demand.csv', index=False)
print("\nSaved -> data/daily_demand.csv")


## 2. Exploratory Data Analysis

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 2: Exploratory Data Analysis
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('data/daily_demand.csv', parse_dates=['date'])
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month_name()
df['year'] = df['date'].dt.year

# ------------------------------------------------------------------
# 1. Overall demand trend
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df['date'], df['Total'], label='Total', color='#4C72B0', linewidth=1)
ax.plot(df['date'], df['City Hotel'], label='City Hotel', color='#DD8452', alpha=0.7, linewidth=1)
ax.plot(df['date'], df['Resort Hotel'], label='Resort Hotel', color='#55A868', alpha=0.7, linewidth=1)
ax.set_title('Daily Room Demand Over Time')
ax.set_ylabel('Rooms Occupied (arrivals)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/01_demand_trend.png')
plt.close()

# ------------------------------------------------------------------
# 2. 7-day rolling average (smooths daily noise, reveals trend)
# ------------------------------------------------------------------
df['total_7d_avg'] = df['Total'].rolling(7).mean()
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df['date'], df['Total'], alpha=0.3, label='Daily', color='gray')
ax.plot(df['date'], df['total_7d_avg'], label='7-day rolling avg', color='#C44E52', linewidth=2)
ax.set_title('Total Demand: Daily vs 7-Day Rolling Average')
ax.legend()
plt.tight_layout()
plt.savefig('plots/02_rolling_average.png')
plt.close()

# ------------------------------------------------------------------
# 3. Day-of-week seasonality
# ------------------------------------------------------------------
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df, x='day_of_week', y='Total', order=dow_order, ax=ax, color='#4C72B0')
ax.set_title('Demand Distribution by Day of Week')
plt.tight_layout()
plt.savefig('plots/03_day_of_week.png')
plt.close()

# ------------------------------------------------------------------
# 4. Monthly seasonality
# ------------------------------------------------------------------
month_order = ['January','February','March','April','May','June','July',
                'August','September','October','November','December']
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=df, x='month', y='Total', order=month_order, ax=ax, color='#55A868')
ax.set_title('Demand Distribution by Month')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('plots/04_monthly_seasonality.png')
plt.close()

# ------------------------------------------------------------------
# 5. Summary stats
# ------------------------------------------------------------------
print("Overall demand stats:")
print(df['Total'].describe())
print("\nAverage demand by day of week:")
print(df.groupby('day_of_week')['Total'].mean().reindex(dow_order).round(1))
print("\nPlots saved to plots/ directory")


## 3. Feature Engineering (Lags, Rolling Stats, Calendar Features)

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 3: Feature Engineering (lag features, rolling stats, calendar features)

We forecast 'Total' daily room demand (across both hotel types).
"""
import pandas as pd
import numpy as np

df = pd.read_csv('data/daily_demand.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

TARGET = 'Total'

# ------------------------------------------------------------------
# 1. Calendar features
# ------------------------------------------------------------------
df['day_of_week'] = df['date'].dt.dayofweek       # 0=Monday
df['day_of_month'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# cyclical encoding so the model understands e.g. Dec(12) is close to Jan(1)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# ------------------------------------------------------------------
# 2. Lag features (yesterday, 2 days ago, same day last week)
# ------------------------------------------------------------------
for lag in [1, 2, 3, 7, 14]:
    df[f'lag_{lag}'] = df[TARGET].shift(lag)

# ------------------------------------------------------------------
# 3. Rolling window statistics (based only on past values, no leakage)
# ------------------------------------------------------------------
df['rolling_mean_7'] = df[TARGET].shift(1).rolling(7).mean()
df['rolling_std_7'] = df[TARGET].shift(1).rolling(7).std()
df['rolling_mean_14'] = df[TARGET].shift(1).rolling(14).mean()
df['rolling_mean_30'] = df[TARGET].shift(1).rolling(30).mean()

# ------------------------------------------------------------------
# 4. Drop rows with NaN from lag/rolling creation (first ~30 days)
# ------------------------------------------------------------------
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with NaN (from lag/rolling window warm-up)")
print(f"Final shape: {df.shape}")

df.to_csv('data/demand_featured.csv', index=False)
print("Saved -> data/demand_featured.csv")


## 4. Preprocessing (Chronological Split, Scaling)

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 4: Preprocessing (chronological train/test split, scaling)

IMPORTANT: time series data must NOT be shuffled before splitting -
we train on the past and test on the future, exactly as a real
deployment would work.
"""
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('data/demand_featured.csv', parse_dates=['date'])

TARGET = 'Total'
feature_cols = [c for c in df.columns if c not in ['date', 'Total', 'City Hotel', 'Resort Hotel',
                                                     'total_7d_avg', 'day_of_week', 'month']]
# (day_of_week/month kept only in their sin/cos + is_weekend form to avoid duplicate raw+cyclical signal)

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df[TARGET]
dates = df['date']

# ------------------------------------------------------------------
# 1. Chronological 80/20 split (last 20% of days = test set)
# ------------------------------------------------------------------
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_train, dates_test = dates.iloc[:split_idx], dates.iloc[split_idx:]

print(f"\nTrain period: {dates_train.min().date()} to {dates_train.max().date()} ({len(X_train)} days)")
print(f"Test period:  {dates_test.min().date()} to {dates_test.max().date()} ({len(X_test)} days)")

# ------------------------------------------------------------------
# 2. Scale features (fit only on train, apply to test - no leakage)
# ------------------------------------------------------------------
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# ------------------------------------------------------------------
# 3. Save
# ------------------------------------------------------------------
np.save('data/X_train_scaled.npy', X_train_scaled)
np.save('data/X_test_scaled.npy', X_test_scaled)
np.save('data/y_train_scaled.npy', y_train_scaled)
np.save('data/y_test_scaled.npy', y_test_scaled)
X_train.to_csv('data/X_train_raw.csv', index=False)
X_test.to_csv('data/X_test_raw.csv', index=False)
y_train.to_csv('data/y_train_raw.csv', index=False)
y_test.to_csv('data/y_test_raw.csv', index=False)
dates_test.to_csv('data/dates_test.csv', index=False)

joblib.dump(scaler_X, 'models/scaler_X.pkl')
joblib.dump(scaler_y, 'models/scaler_y.pkl')
joblib.dump(feature_cols, 'models/feature_columns.pkl')

print("\nSaved preprocessed data and scalers")


## 5. Baseline Models (Naive + Linear Regression + Random Forest)

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 5: Baseline Models (Linear Regression + Random Forest)
"""
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train_raw = pd.read_csv('data/X_train_raw.csv')
X_test_raw = pd.read_csv('data/X_test_raw.csv')
y_train = pd.read_csv('data/y_train_raw.csv').values.ravel()
y_test = pd.read_csv('data/y_test_raw.csv').values.ravel()
dates_test = pd.read_csv('data/dates_test.csv', parse_dates=['date'])['date']

results = {}

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    r2 = r2_score(y_true, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}
    print(f"\n--- {name} ---")
    for k, v in results[name].items():
        print(f"  {k}: {v:.3f}")

# ------------------------------------------------------------------
# 1. Naive baseline: "tomorrow = today" (lag_1) - the sanity-check floor
# ------------------------------------------------------------------
naive_pred = X_test_raw['lag_1'].values
evaluate('Naive (yesterday = today)', y_test, naive_pred)

# ------------------------------------------------------------------
# 2. Linear Regression
# ------------------------------------------------------------------
lr = LinearRegression()
lr.fit(X_train_raw, y_train)
lr_pred = lr.predict(X_test_raw)
evaluate('Linear Regression', y_test, lr_pred)

# ------------------------------------------------------------------
# 3. Random Forest
# ------------------------------------------------------------------
rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train_raw, y_train)
rf_pred = rf.predict(X_test_raw)
evaluate('Random Forest', y_test, rf_pred)

# ------------------------------------------------------------------
# 4. Plot actual vs predicted (test period)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dates_test, y_test, label='Actual', color='black', linewidth=1.5)
ax.plot(dates_test, lr_pred, label='Linear Regression', alpha=0.7)
ax.plot(dates_test, rf_pred, label='Random Forest', alpha=0.7)
ax.set_title('Room Demand Forecast: Actual vs Predicted (Test Period)')
ax.set_ylabel('Total Room Demand')
ax.legend()
plt.tight_layout()
plt.savefig('plots/05_baseline_forecast_comparison.png')
plt.close()

# ------------------------------------------------------------------
# 5. Feature importance (RF)
# ------------------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X_train_raw.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='#55A868')
ax.invert_yaxis()
ax.set_title('Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('plots/06_feature_importance.png')
plt.close()

# ------------------------------------------------------------------
# 6. Save results & models
# ------------------------------------------------------------------
results_df = pd.DataFrame(results).T
results_df.to_csv('models/baseline_results.csv')
joblib.dump(lr, 'models/linear_regression.pkl')
joblib.dump(rf, 'models/random_forest.pkl')

print("\n\nComparison table:")
print(results_df.round(3))


## 6. Deep Learning Model (LSTM — requires TensorFlow, run in Colab)

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 6: Deep Learning Model (LSTM sequence-to-one forecaster)

>>> RUN THIS IN GOOGLE COLAB OR A LOCAL ENVIRONMENT WITH TENSORFLOW INSTALLED <<<
(pip install tensorflow)

Unlike the baseline models (which use flattened lag features), this LSTM
consumes a genuine sliding window of the last N days as a sequence, letting
it learn temporal patterns directly rather than through hand-crafted lags.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tf.random.set_seed(42)
np.random.seed(42)

WINDOW_SIZE = 14   # use the last 14 days to predict the next day

# ------------------------------------------------------------------
# 1. Load the daily demand series (built in 01_build_timeseries.py)
# ------------------------------------------------------------------
df = pd.read_csv('data/daily_demand.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

# also bring in calendar features for a multivariate window
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

feature_cols = ['Total', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
data = df[feature_cols].values

# ------------------------------------------------------------------
# 2. Chronological split BEFORE scaling (fit scaler on train only)
# ------------------------------------------------------------------
split_idx = int(len(df) * 0.8)
train_data = data[:split_idx]
test_data = data[split_idx - WINDOW_SIZE:]   # include lookback context for first test window

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# ------------------------------------------------------------------
# 3. Build sliding windows: X = [t-14..t-1] all features, y = Total at t
# ------------------------------------------------------------------
def make_windows(arr, window_size, target_col_idx=0):
    X, y = [], []
    for i in range(window_size, len(arr)):
        X.append(arr[i-window_size:i])
        y.append(arr[i, target_col_idx])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_scaled, WINDOW_SIZE)
X_test, y_test = make_windows(test_scaled, WINDOW_SIZE)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# ------------------------------------------------------------------
# 4. Build the LSTM
# ------------------------------------------------------------------
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(feature_cols))),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)   # regression output: scaled demand
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

# ------------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=callbacks,
    verbose=1,
    shuffle=False   # preserve temporal order even within training
)

# ------------------------------------------------------------------
# 6. Predict & inverse-transform back to actual room counts
# ------------------------------------------------------------------
y_pred_scaled = model.predict(X_test).ravel()

# inverse transform: reconstruct full feature vector shape for the scaler
def inverse_target(scaled_target, scaler, target_col_idx=0, n_features=5):
    dummy = np.zeros((len(scaled_target), n_features))
    dummy[:, target_col_idx] = scaled_target
    return scaler.inverse_transform(dummy)[:, target_col_idx]

y_pred = inverse_target(y_pred_scaled, scaler, n_features=len(feature_cols))
y_true = inverse_target(y_test, scaler, n_features=len(feature_cols))

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
r2 = r2_score(y_true, y_pred)

print("\n--- LSTM Test Set Performance (original scale) ---")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.3f}%")
print(f"R2:   {r2:.3f}")

# ------------------------------------------------------------------
# 7. Plot actual vs predicted
# ------------------------------------------------------------------
test_dates = df['date'].iloc[split_idx:split_idx + len(y_true)].values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_true, label='Actual', color='black', linewidth=1.5)
ax.plot(test_dates, y_pred, label='LSTM Prediction', color='#8172B3', alpha=0.8)
ax.set_title('LSTM Room Demand Forecast: Actual vs Predicted (Test Period)')
ax.set_ylabel('Total Room Demand')
ax.legend()
plt.tight_layout()
plt.savefig('plots/07_lstm_forecast.png')
plt.close()

# ------------------------------------------------------------------
# 8. Training curves
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('LSTM Training Loss (MSE, scaled)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/08_lstm_training_curve.png')
plt.close()

# ------------------------------------------------------------------
# 9. Save model & scaler
# ------------------------------------------------------------------
model.save('models/lstm_demand_model.keras')
import joblib
joblib.dump(scaler, 'models/lstm_scaler.pkl')
print("\nModel saved -> models/lstm_demand_model.keras")


## 7. Improved LSTM — Detrended Residual Modeling (requires TensorFlow, run in Colab)

In [ ]:
"""
Module 3: Room Demand Forecasting
Step 6b: Improved LSTM — models the DETRENDED residual, not raw demand

>>> RUN THIS IN GOOGLE COLAB OR A LOCAL ENVIRONMENT WITH TENSORFLOW INSTALLED <<<

Why the first LSTM struggled: the daily demand series has a clear upward
trend across the ~2 years of data. A scaler fit on the training period alone
means the test period's demand sits at a systematically different level than
what the model learned - collapsing R2 even when relative day-to-day and
weekly patterns are captured correctly.

Fix: instead of predicting raw demand, we predict the RESIDUAL from a
trailing 14-day local average (a quantity that's naturally much more
stationary), then add that local average back at the end. The local average
uses only past days, so this is fully causal - no leakage.

We also shrink the network (single LSTM layer, more dropout) since ~600
training days is a small sample for a deep sequence model.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tf.random.set_seed(42)
np.random.seed(42)

WINDOW_SIZE = 14

# ------------------------------------------------------------------
# 1. Load data & build the trailing local average (causal - uses only
#    days strictly before t, so it's known and valid at prediction time)
# ------------------------------------------------------------------
df = pd.read_csv('data/daily_demand.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

df['local_avg_14'] = df['Total'].shift(1).rolling(14).mean()
df['residual'] = df['Total'] - df['local_avg_14']

df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df = df.dropna().reset_index(drop=True)   # drop the first 14 warm-up days
print(f"Usable rows after computing local average: {len(df)}")

feature_cols = ['residual', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
data = df[feature_cols].values

# ------------------------------------------------------------------
# 2. Chronological split, scale (fit on train only)
# ------------------------------------------------------------------
split_idx = int(len(df) * 0.8)
train_data = data[:split_idx]
test_data = data[split_idx - WINDOW_SIZE:]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

def make_windows(arr, window_size, target_col_idx=0):
    X, y = [], []
    for i in range(window_size, len(arr)):
        X.append(arr[i-window_size:i])
        y.append(arr[i, target_col_idx])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_scaled, WINDOW_SIZE)
X_test, y_test = make_windows(test_scaled, WINDOW_SIZE)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# ------------------------------------------------------------------
# 3. Smaller network, more regularization (limited data)
# ------------------------------------------------------------------
model = Sequential([
    LSTM(32, input_shape=(WINDOW_SIZE, len(feature_cols)), kernel_regularizer=l2(1e-4)),
    Dropout(0.3),
    Dense(16, activation='relu', kernel_regularizer=l2(1e-4)),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=16,
    callbacks=callbacks,
    verbose=1,
    shuffle=False
)

# ------------------------------------------------------------------
# 4. Predict residuals, inverse-transform, then ADD BACK the local average
# ------------------------------------------------------------------
y_pred_scaled = model.predict(X_test).ravel()

def inverse_target(scaled_target, scaler, n_features):
    dummy = np.zeros((len(scaled_target), n_features))
    dummy[:, 0] = scaled_target
    return scaler.inverse_transform(dummy)[:, 0]

residual_pred = inverse_target(y_pred_scaled, scaler, len(feature_cols))
residual_true = inverse_target(y_test, scaler, len(feature_cols))

# the local averages aligned with the test predictions
local_avgs_test = df['local_avg_14'].iloc[split_idx: split_idx + len(residual_pred)].values

y_pred = residual_pred + local_avgs_test
y_true = residual_true + local_avgs_test   # should equal original Total values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
r2 = r2_score(y_true, y_pred)

print("\n--- Improved LSTM (detrended residual) Test Performance ---")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.3f}%")
print(f"R2:   {r2:.3f}")

# ------------------------------------------------------------------
# 5. Plot
# ------------------------------------------------------------------
test_dates = df['date'].iloc[split_idx: split_idx + len(y_true)].values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_true, label='Actual', color='black', linewidth=1.5)
ax.plot(test_dates, y_pred, label='Improved LSTM Prediction', color='#8172B3', alpha=0.8)
ax.plot(test_dates, local_avgs_test, label='14-day Local Average (reference)', color='gray', linestyle='--', alpha=0.6)
ax.set_title('Improved LSTM Forecast (Detrended Residual Model)')
ax.set_ylabel('Total Room Demand')
ax.legend()
plt.tight_layout()
plt.savefig('plots/09_lstm_improved_forecast.png')
plt.close()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('Improved LSTM Training Loss (MSE, scaled residual)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/10_lstm_improved_training_curve.png')
plt.close()

model.save('models/lstm_improved_demand_model.keras')
import joblib
joblib.dump(scaler, 'models/lstm_improved_scaler.pkl')
print("\nModel saved -> models/lstm_improved_demand_model.keras")
